<a href="https://colab.research.google.com/github/harshitnagar22/Celebal_CEI/blob/main/week6_harshit_nagar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 6 Assessment: MNIST Image Denoising using Autoencoders
### Project 1: Build a deep learning model to remove noise from images using an autoencoder on MNIST

In this notebook, I have built a Denoising Autoencoder to clean noisy handwritten digit images from the MNIST dataset. I implemented and compared two different models:
1. **Fully Connected (Dense) Autoencoder** - Simple baseline model that flattens the image.
2. **Convolutional Autoencoder (CAE)** - Advanced model that keeps the 2D image structure using convolution layers.

**Steps followed:**
- Load and normalize the MNIST dataset from local PNG folders (fallback to Keras if local path not found).
- Add random Gaussian noise to create corrupted input images.
- Train both models to reconstruct the clean images from the noisy ones.
- Evaluate performance visually and quantitatively using MSE and PSNR (Peak Signal-to-Noise Ratio).

In [ ]:
import os
import glob
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)

## 1. Load and Preprocess Dataset
Here I load the MNIST dataset from the local `mnist_png` folder. To make it work in any environment (like Colab), I wrote a check: if the local folder isn't there, it automatically downloads from `keras.datasets`. 

We divide the pixel values by 255.0 to scale them between 0 and 1, and add a channel dimension so it works with the convolution layers.

In [ ]:
def load_data(data_dir='mnist_png'):
    # Check if local folder exists
    if os.path.exists(data_dir):
        print(f"Loading dataset from local folder: {data_dir}")
        
        def load_split(split):
            path = os.path.join(data_dir, split)
            images, labels = [], []
            file_paths = glob.glob(os.path.join(path, '*', '*.png'))
            print(f"Found {len(file_paths)} images in {split}.")
            
            for f in file_paths:
                img = Image.open(f).convert('L')
                label = int(os.path.basename(os.path.dirname(f)))
                images.append(np.array(img))
                labels.append(label)
                
            return np.array(images), np.array(labels)
        
        x_train, y_train = load_split('training')
        x_test, y_test = load_split('testing')
    else:
        print("Local folders not found, loading from tf.keras...")
        (x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
    
    # Scaling pixels to [0, 1]
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0
    
    # Reshape for CNN layers (batch, 28, 28, 1)
    x_train = np.expand_dims(x_train, axis=-1)
    x_test = np.expand_dims(x_test, axis=-1)
    
    print(f"Train images shape: {x_train.shape}, labels shape: {y_train.shape}")
    print(f"Test images shape: {x_test.shape}, labels shape: {y_test.shape}")
    return x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = load_data()

## 2. Add Artificial Noise
To create the noisy input images, I am adding random Gaussian noise with a standard deviation (`noise_factor`) of `0.5`. Since adding noise might make pixel values go below 0 or above 1, we use `np.clip` to keep them inside the $[0, 1]$ range.

In [ ]:
noise_factor = 0.5

# Add normal distribution noise
x_train_noisy = x_train + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_train.shape)
x_test_noisy = x_test + noise_factor * np.random.normal(loc=0.0, scale=1.0, size=x_test.shape)

# Keep pixel values between 0 and 1
x_train_noisy = np.clip(x_train_noisy, 0.0, 1.0)
x_test_noisy = np.clip(x_test_noisy, 0.0, 1.0)

print("Noisy training set shape:", x_train_noisy.shape)
print("Noisy test set shape:", x_test_noisy.shape)

### Visualizing Clean vs Noisy Images
Let's print some test images side by side to see how noisy they look.

In [ ]:
n = 8
plt.figure(figsize=(16, 4))
for i in range(n):
    # Clean original
    ax = plt.subplot(2, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    plt.title(f"Label: {y_test[i]}")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)
    
    # Corrupted noisy
    ax = plt.subplot(2, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    plt.title("Noisy")
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

plt.suptitle("Clean vs Noisy Images (Test Set)", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Build Autoencoder Models

### Model 1: Fully Connected Autoencoder (Baseline)
For the baseline, we flatten the 28x28 image to a 784-length vector. We then compress it down to a hidden layer size of 128, then to a 64 bottleneck layer. The decoder maps it back to 128, then to 784, and we reshape it to 28x28.

In [ ]:
def build_dense_model():
    # Encoder part
    inputs = layers.Input(shape=(28, 28, 1), name="encoder_input")
    x = layers.Flatten()(inputs)
    x = layers.Dense(128, activation='relu')(x)
    bottleneck = layers.Dense(64, activation='relu', name="bottleneck")(x)
    
    # Decoder part
    x = layers.Dense(128, activation='relu')(bottleneck)
    x = layers.Dense(784, activation='sigmoid')(x)
    outputs = layers.Reshape((28, 28, 1), name="decoder_output")(x)
    
    model = models.Model(inputs, outputs, name="Dense_Autoencoder")
    return model

dense_autoencoder = build_dense_model()
dense_autoencoder.summary()

### Model 2: Convolutional Autoencoder
Flattening the image loses spatial information. To fix this, I built a Convolutional Autoencoder (CAE). 
- **Encoder**: Uses Conv2D filters to extract 2D patterns and MaxPooling2D to downsample the image size.
- **Decoder**: Uses Conv2DTranspose (transposed convolutions) to upsample the features back to 28x28.

In [ ]:
def build_conv_model():
    # Encoder
    inputs = layers.Input(shape=(28, 28, 1), name="conv_encoder_input")
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D((2, 2), padding='same')(x)
    x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(x)
    bottleneck = layers.MaxPooling2D((2, 2), padding='same', name="conv_bottleneck")(x)
    
    # Decoder
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(bottleneck)
    x = layers.Conv2DTranspose(32, (3, 3), strides=2, activation='relu', padding='same')(x)
    outputs = layers.Conv2D(1, (3, 3), activation='sigmoid', padding='same', name="conv_decoder_output")(x)
    
    model = models.Model(inputs, outputs, name="Conv_Autoencoder")
    return model

conv_autoencoder = build_conv_model()
conv_autoencoder.summary()

## 4. Train the Models
We use standard binary cross-entropy as the loss function since the image pixel values are normalized to $[0, 1]$. We compile both models with the Adam optimizer and use an early stopping callback with a patience of 3 so it doesn't overfit.

In [ ]:
epochs = 15
batch_size = 128

# Stop if validation loss stops improving
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

### Train Model 1 (Dense Autoencoder)

In [ ]:
dense_autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

print("Training Fully Connected Autoencoder...")
dense_history = dense_autoencoder.fit(
    x_train_noisy, x_train,
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    callbacks=[early_stop]
)

### Train Model 2 (Convolutional Autoencoder)

In [ ]:
conv_autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

print("Training Convolutional Autoencoder...")
conv_history = conv_autoencoder.fit(
    x_train_noisy, x_train,
    epochs=epochs,
    batch_size=batch_size,
    shuffle=True,
    validation_data=(x_test_noisy, x_test),
    callbacks=[early_stop]
)

### Compare Loss Curves

In [ ]:
plt.figure(figsize=(14, 5))

# Dense Loss
plt.subplot(1, 2, 1)
plt.plot(dense_history.history['loss'], label='Train Loss')
plt.plot(dense_history.history['val_loss'], label='Val Loss')
plt.title('Dense Autoencoder Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

# Conv Loss
plt.subplot(1, 2, 2)
plt.plot(conv_history.history['loss'], label='Train Loss')
plt.plot(conv_history.history['val_loss'], label='Val Loss')
plt.title('Convolutional Autoencoder Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid(True)

plt.suptitle('Training vs Validation Loss')
plt.tight_layout()
plt.show()

## 5. Denoise Outputs and Calculate Metrics
Now we clean the test images using both trained models. To measure how well they did quantitatively, we calculate:
1. **Mean Squared Error (MSE)** - lower is better.
2. **Peak Signal-to-Noise Ratio (PSNR)** - higher is better, calculated as:
   $$\text{PSNR} = 10 \cdot \log_{10} \left( \frac{1.0}{\text{MSE}} \right)$$

In [ ]:
# Predict reconstructed denoised images
dense_pred = dense_autoencoder.predict(x_test_noisy)
conv_pred = conv_autoencoder.predict(x_test_noisy)

def get_psnr(target, pred):
    mse = np.mean((target - pred) ** 2)
    if mse == 0:
        return float('inf')
    return 10 * np.log10(1.0 / mse)

# Calculate MSE
noisy_mse = np.mean((x_test - x_test_noisy) ** 2)
dense_mse = np.mean((x_test - dense_pred) ** 2)
conv_mse = np.mean((x_test - conv_pred) ** 2)

# Calculate PSNR
noisy_psnr = get_psnr(x_test, x_test_noisy)
dense_psnr = get_psnr(x_test, dense_pred)
conv_psnr = get_psnr(x_test, conv_pred)

metrics_df = pd.DataFrame({
    'Metric': ['MSE', 'PSNR (dB)'],
    'Noisy Image (Baseline)': [noisy_mse, noisy_psnr],
    'Dense Autoencoder': [dense_mse, dense_psnr],
    'Convolutional Autoencoder': [conv_mse, conv_psnr]
})

print(metrics_df.to_string(index=False))

## 6. Visual Comparison of Results
Let's plot clean, noisy, and the denoised outputs side-by-side to visually inspect the difference.

In [ ]:
n = 10
plt.figure(figsize=(20, 8))
for i in range(n):
    # 1. Clean original
    ax = plt.subplot(4, n, i + 1)
    plt.imshow(x_test[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.ylabel("Clean Target", fontsize=12)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_ticks([])
    
    # 2. Noisy inputs
    ax = plt.subplot(4, n, i + 1 + n)
    plt.imshow(x_test_noisy[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.ylabel("Noisy Input", fontsize=12)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_ticks([])
    
    # 3. Dense reconstructions
    ax = plt.subplot(4, n, i + 1 + 2*n)
    plt.imshow(dense_pred[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.ylabel("Dense Denoised", fontsize=12)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_ticks([])
    
    # 4. Conv reconstructions
    ax = plt.subplot(4, n, i + 1 + 3*n)
    plt.imshow(conv_pred[i].reshape(28, 28), cmap='gray')
    if i == 0:
        plt.ylabel("Conv Denoised", fontsize=12)
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_ticks([])

plt.suptitle("Denoising Results (Comparison Grid)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 7. Observations and Analysis

Here are my observations from running this assignment:

1. **Dense vs. Convolutional Performance**:
   - The **Fully Connected (Dense) Autoencoder** succeeds in cleaning up the random noise, but the results look very **blurry** and soft. This is because flattening the 2D image `(28x28)` into a 1D vector `(784)` completely discards spatial correlations, and the Dense layers struggle to recreate clean line details. It gets a PSNR of `~18.17 dB`.
   - The **Convolutional Autoencoder** produces much **sharper** images. The strokes of the digits are well-defined, and the background remains clean. By using Conv2D layers, it processes the image in 2D and retains spatial patterns. It achieves a PSNR of `~21.18 dB` (which is about a `+3.0 dB` improvement over the Dense model).

2. **Parameter Count Comparison**:
   - What is interesting is that the CNN model uses only **28,353 parameters**, while the Fully Connected baseline model uses **218,192 parameters**. The CAE uses about $87\%$ fewer weights but performs much better, demonstrating that CNN architectures are far more efficient for image data.

3. **Compression & Noise Filtering**:
   - By forcing the input through a small bottleneck representation (64 units for Dense, and $7 \times 7 \times 32$ for CNN), the autoencoders act as a filter. Since random noise is chaotic and has no coherent structure, the models can't compress it. Instead, they only learn to compress and reconstruct the main underlying structure (the digit shape).

4. **Conclusion**:
   - Convolutional structures are much more suitable for image denoising. While both models remove the noise, the Convolutional Autoencoder is superior in both reconstruction quality and parameter efficiency.